# Banking Logistic Regression – Practice Skeleton
**Domain:** Credit Risk / Loan Underwriting  
**Source pattern:** Coursera ML C1_W3 Logistic Regression lab, adapted to banking

A regional bank wants to:
1. **Approve / decline personal loans** using two underwriting features (Credit Score proxy + DTI-related score) — roughly linear separation.
2. **Score more complex credit products** where two risk indicators form a non-linear “acceptable risk” region — requires polynomial feature mapping + L2 regularization.

Use this notebook to practice. The companion **Solution** notebook contains complete answers, vectorized alternates, extra practice, and a policy-simulation section.

### Learning Objectives
- Implement sigmoid, binary cross-entropy cost and gradients from scratch
- Train a linear logistic model for loan approval and plot the decision boundary
- Apply feature mapping + regularization for non-linear credit-risk boundaries
- Translate model probability into a risk-appetite threshold (approve / reject policy)
- Communicate results to Credit Risk, Underwriting, and Executive audiences


## Cheat-Sheet (keep open)
See also **Banking_Logistic_Regression_Cheatsheet.docx**.

| Item | Banking meaning / Code |
|------|------------------------|
| Sigmoid | Maps linear score → P(approve) or P(low risk) |
| Cost | Average “surprise” of the model’s probability vs actual decision |
| λ (lambda) | Regularization strength — higher λ = smoother, more conservative boundary |
| Threshold | Risk appetite lever (0.5 default; lower = more approvals / higher volume) |
| Feature map | Expands two risk scores into polynomial terms so a linear classifier can capture curved risk regions |


## 0. Packages


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy
import math

%matplotlib inline
print("Libraries ready")


## 1. Linear Logistic Regression – Loan Approval

### 1.1 Load & explore
`credit_score` and `dti_ratio` are the two underwriting features.  
`approved` = 1 (loan granted) or 0 (declined).


In [ ]:
# TODO: load the data
# df = pd.read_csv("data/bank_loan_approval.csv")
# X_train = df[["credit_score", "dti_ratio"]].values
# y_train = df["approved"].values

# YOUR CODE HERE


print("Shape of X_train:", X_train.shape)
print("Approval rate: {:.1%}".format(y_train.mean()))
print("First 5 rows:\n", np.column_stack((X_train[:5], y_train[:5])))


### 1.2 Visualise the underwriting space


In [ ]:
# TODO: scatter plot – black + for Approved, gold circles for Declined
# Add axis labels and a legend

# YOUR CODE HERE


### 1.3 Sigmoid – maps underwriting score to probability of approval
$$g(z) = \frac{1}{1+e^{-z}}$$


In [ ]:
def sigmoid(z):
    """Compute sigmoid of z (scalar or array). Clip for numerical stability."""
    ### START CODE HERE ###
    
    
    ### END CODE HERE ###
    return g

print("sigmoid(0) =", sigmoid(0))   # expect 0.5
print("sigmoid([-2,0,2]) =", sigmoid(np.array([-2.,0,2])))


### 1.4 Cost function (binary cross-entropy)
Measures how well the model’s predicted probabilities match the historical approve/decline decisions.


In [ ]:
def compute_cost(X, y, w, b, *argv):
    """Logistic regression cost (non-regularized)"""
    m = X.shape[0]
    ### START CODE HERE ###
    
    
    ### END CODE HERE ###
    return total_cost

print("Cost at zero weights (should be ~0.693):", 
      compute_cost(X_train, y_train, np.zeros(2), 0.))


### 1.5 Gradients


In [ ]:
def compute_gradient(X, y, w, b, *argv):
    """Return dj_db (scalar), dj_dw (vector)"""
    m, n = X.shape
    dj_dw = np.zeros(w.shape)
    dj_db = 0.
    ### START CODE HERE ###
    
    
    ### END CODE HERE ###
    return dj_db, dj_dw


### 1.6 Gradient Descent – learn the underwriting parameters


In [ ]:
def gradient_descent(X, y, w_in, b_in, cost_fn, grad_fn, alpha, num_iters, lambda_=0):
    w = copy.deepcopy(w_in).astype(float)
    b = float(b_in)
    J_history = []
    for i in range(num_iters):
        dj_db, dj_dw = grad_fn(X, y, w, b, lambda_)
        w -= alpha * dj_dw
        b -= alpha * dj_db
        if i % max(1, num_iters//10) == 0 or i == num_iters-1:
            cost = cost_fn(X, y, w, b, lambda_)
            J_history.append(cost)
            print(f"Iteration {i:6d}: Cost {cost:.4f}")
    return w, b, J_history

# TODO: run GD with alpha ≈ 0.001 and many iterations (e.g. 100_000)
# initial_w = np.zeros(2); initial_b = 0.
# w, b, J_hist = gradient_descent(...)

# YOUR CODE HERE

print("Learned w, b =", w, b)


### 1.7 Decision boundary & accuracy
The line where P(approve) = 0.5 is the bank’s current policy boundary.


In [ ]:
def predict(X, w, b, threshold=0.5):
    """Binary approve/decline decisions"""
    ### START CODE HERE ###
    
    
    ### END CODE HERE ###
    return p

# TODO: plot data + decision boundary, compute training accuracy

# YOUR CODE HERE


## 2. Regularized Logistic Regression – Complex Credit Risk

Two derived risk scores form a non-linear “acceptable risk” region.  
We expand them with polynomial features (degree 6) and add L2 regularization so the policy boundary stays smooth and generalizable.


In [ ]:
# Load complex-risk data
df2 = pd.read_csv("data/bank_complex_risk.csv")
X_risk = df2[["risk_score_A", "risk_score_B"]].values
y_risk = df2["approved"].values

plt.figure(figsize=(6,5))
pos = y_risk == 1
plt.scatter(X_risk[pos,0], X_risk[pos,1], c='k', marker='+', label='Acceptable risk')
plt.scatter(X_risk[~pos,0], X_risk[~pos,1], c='gold', marker='o', edgecolors='k', label='High risk / Decline')
plt.xlabel("Risk Score A"); plt.ylabel("Risk Score B")
plt.title("Complex Credit Product – Non-linear Risk Pattern"); plt.legend(); plt.show()


### 2.1 Feature mapping


In [ ]:
def map_feature(X1, X2, degree=6):
    """Polynomial feature map (degree 6 → 27 features)"""
    X1 = np.atleast_1d(X1)
    X2 = np.atleast_1d(X2)
    out = []
    for i in range(1, degree+1):
        for j in range(i+1):
            out.append((X1**(i-j) * (X2**j)))
    return np.stack(out, axis=1)

X_mapped = map_feature(X_risk[:,0], X_risk[:,1])
print("Mapped shape (expect ~ (118, 27)):", X_mapped.shape)


### 2.2 Regularized cost & gradient
Add $$\frac{\lambda}{2m}\|w\|^2$$ to the cost and $$\frac{\lambda}{m}w_j$$ to each weight gradient.  
**Do not regularize the bias.**


In [ ]:
def compute_cost_reg(X, y, w, b, lambda_=1):
    ### START CODE HERE ###
    
    
    ### END CODE HERE ###
    return total_cost

def compute_gradient_reg(X, y, w, b, lambda_=1):
    ### START CODE HERE ###
    
    
    ### END CODE HERE ###
    return dj_db, dj_dw


### 2.3 Train, plot boundary, evaluate
Try λ = 1 first (classic value). Then experiment with 0.01 and 10.


In [ ]:
# TODO: initialise, run GD, plot non-linear boundary, print accuracy
# np.random.seed(1)
# initial_w = np.random.rand(X_mapped.shape[1]) - 0.5
# initial_b = 1.0
# w_reg, b_reg, _ = gradient_descent(X_mapped, y_risk, initial_w, initial_b,
#                                    compute_cost_reg, compute_gradient_reg,
#                                    alpha=0.01, num_iters=10000, lambda_=1.0)

# YOUR CODE HERE


## 3. Alternate Implementations
- Fully vectorized cost / gradient (no Python loops)
- `sklearn.linear_model.LogisticRegression` (compare coefficients & accuracy)


In [ ]:
# YOUR CODE HERE – vectorized or sklearn alternate


## 4. More Practice
1. Change the decision threshold from 0.5 → 0.3 (more volume) and 0.7 (more conservative). How does approval volume change?
2. Predict approval probability for an applicant with credit_score=55, dti_ratio=70.
3. Retrain the complex-risk model with λ = 0. Does the boundary look over-fit?
4. Flip 10 % of the labels at random and retrain. What happens to accuracy and boundary smoothness?


In [ ]:
# Practice experiments
# YOUR CODE HERE


## 5. Simulation Lab – Policy Levers
Explore how λ (model complexity) and the decision threshold affect the volume-versus-risk trade-off.


In [ ]:
# Simulation parameters – change these and re-run
LAMBDA_GRID = [0.0, 0.01, 0.1, 1.0, 10.0]
THRESHOLD   = 0.5
ITERS       = 8000
ALPHA       = 0.01

# YOUR CODE HERE – train for each λ, record accuracy, plot curve


## 6. Audience-Adapted Messaging
(Reference: supplied audience-analysis PDFs)

- **Credit Risk / Model Validation team** – show cost curves, gradient formulas, λ sensitivity, residual diagnostics.
- **Underwriting managers** – focus on the decision-boundary plots and the practical meaning of the threshold.
- **Risk Committee / Executives** – one chart + one sentence: “With λ=1 the model correctly classifies ~82 % of historic complex-risk cases; raising λ produces a smoother, more conservative boundary at a modest accuracy cost.”

Write a 3-sentence executive summary of your results:


In [ ]:
# Executive summary
print(""" 
...
""")


## 7. Key Takeaways (fill after finishing)
- 
- 
-
